# EduLogGen v1.6.0: GRU and Transformer tuning on a Colab GPU

Runs the equal-budget tuning protocol of `studies/tune_v16.py` (read its docstring for the full protocol):

1. **search**: 12 configurations per model, training seed 0, scored on inner-validation learners;
2. **confirm**: the 3 best configurations, training seeds 0, 1, 2;
3. **final**: the winner and the untuned defaults, training seeds 0, 1, 2, scored on the holdout learners.

The GPU speeds up **training** only. Generation and scoring run on the CPU, by design.

**Before you start**

- Runtime → Change runtime type → **GPU** (a T4 is enough).
- Put `study-data.zip` in `MyDrive/eduloggen/`. On your Mac, create it from the repository root:
  `cd studies && zip -r ~/Downloads/study-data.zip oulad/data ednet/data`
  It contains the prepared OULAD modules and the EdNet-KT4 sample. EdNet is licensed CC BY-NC 4.0: keep the zip in your own Drive and don't share it.
- Results are written to `MyDrive/eduloggen/tuning/` after every run. If Colab disconnects, run the notebook again from the top; finished runs are skipped.

In [ ]:
import os

# Fixed cuBLAS workspace for deterministic CUDA kernels; must be set before CUDA starts.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/eduloggen"
RESULTS = f"{BASE}/tuning"
os.makedirs(RESULTS, exist_ok=True)

In [ ]:
%cd /content
![ -d EduLogGen ] || git clone --branch feature/v1.6.0-transformer --depth 1 https://github.com/anisbey1/EduLogGen.git
%cd /content/EduLogGen
!git pull --ff-only
# Colab already ships a CUDA build of PyTorch; install EduLogGen itself without touching it.
!pip install -q pyyaml
!pip install -q --no-deps --force-reinstall .
!unzip -n -q {BASE}/study-data.zip -d studies
!ls studies/oulad/data studies/ednet/data

## GPU check

Fits a small GRU and a small Transformer twice on the GPU and compares the models. GPU training is only best-effort deterministic: if the fingerprints differ, the runs still work, but report GPU results as not bit-reproducible.

In [ ]:
import torch

import eduloggen as elg

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
assert torch.cuda.is_available(), "Select a GPU runtime first"

demo = elg.sessionize(elg.demo_dataset(100, seed=1))
small = {
    "gru": {"embedding_dim": 16, "hidden_dim": 32, "epochs": 5, "device": "cuda"},
    "transformer": {"d_model": 32, "num_heads": 4, "epochs": 5, "device": "cuda"},
}
for name, params in small.items():
    first = elg.fit_generator(name, demo, hyperparameters=params)
    second = elg.fit_generator(name, demo, hyperparameters=params)
    same_output = elg.generate(first, n_sessions=20, seed=1).events == elg.generate(second, n_sessions=20, seed=1).events
    print(f"{name}: identical fits {first.fingerprint() == second.fingerprint()}, identical samples {same_output}")

## Tuning

Each line is one model and one dataset, run stage by stage. On a T4, expect several hours in total, with EdNet the slowest. You can run one cell per session; every command resumes where it stopped.

In [ ]:
import subprocess
import sys


def tune(study, model):
    """Run the three stages in order; stop at the first failure."""
    for stage in ("search", "confirm", "final"):
        command = [sys.executable, "studies/tune_v16.py", "--study", study, "--model", model,
                   "--stage", stage, "--device", "cuda", "--results", RESULTS]
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as run:
            for line in run.stdout:
                print(line, end="")
        if run.returncode:
            raise RuntimeError(f"{stage} failed (exit code {run.returncode})")

In [ ]:
tune("ednet", "transformer")

In [ ]:
tune("ednet", "gru")

In [ ]:
tune("oulad", "transformer")

In [ ]:
tune("oulad", "gru")

## Report

Writes `tuning.md` next to the result files. Bring the `tuning/` folder back to `studies/tuning/` in the repository; the final holdout numbers are GPU-trained and should be reported as such.

In [ ]:
!python studies/tune_v16.py --report --results {RESULTS} > {RESULTS}/tuning.md
from IPython.display import Markdown, display

display(Markdown(open(f"{RESULTS}/tuning.md").read()))